In [ ]:
# EpistemicTrap CCC - Setup
!pip install -q matplotlib seaborn
import os
os.environ['RENDER_SUBRUNS'] = 'False'
import kaggle_benchmarks as kbench
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
MODELS_TO_TEST = [kbench.llm]  # Add more models here for comparative runs


In [ ]:
import glob
json_paths = glob.glob('/kaggle/input/**/*.json', recursive=True)
dataset_path = next((p for p in json_paths if 'metacog_dataset.json' in p), None)
if not dataset_path: raise FileNotFoundError('Could not find metacog_dataset.json in /kaggle/input/')
with open(dataset_path) as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df = df[df['subtype'] == 'CCC'].reset_index(drop=True)
print(f'Loaded CCC data: {len(df)} items.')


In [ ]:
CONFIDENCE_PATTERNS = [
    r"[Cc]onfidence[:\s]+(\d{1,3})%?", r"(\d{1,3})%\s+confident",
    r"[Cc]onfidence\s+[Ss]core[:\s]+(\d{1,3})", r"\bStep\s+\d[:\s]+.*?(\d{1,3})%",
    r"[Cc]ertainty[:\s]+(\d{1,3})%?",
]

def extract_all_confidences(text: str) -> list[float]:
    all_matches = []
    for pattern in CONFIDENCE_PATTERNS:
        for m in re.finditer(pattern, text):
            try:
                val = float(m.group(1))
                if 0 <= val <= 100:
                    all_matches.append((m.start(), val))
            except Exception:
                continue
    all_matches.sort(key=lambda x: x[0])
    deduped = []
    prev_pos = -20
    for pos, val in all_matches:
        if pos - prev_pos > 10:
            deduped.append(val)
            prev_pos = pos
    return deduped

@kbench.task(name='metacog_ccc', version=3)
def test_confidence_calibration(llm, prompt: str, expected_confidence_pattern: str, **kwargs) -> float:
    full_prompt = f"{prompt}\n\n" + \
        "IMPORTANT: At each step of your reasoning, explicitly state your current " + \
        "confidence as a percentage (e.g., 'Confidence: 78%'). Your confidence " + \
        "at each subsequent step should reflect the cumulative uncertainty from " + \
        "all prior steps — do not reset it."
    response = llm.prompt(full_prompt)
    confidences = extract_all_confidences(response)
    if len(confidences) < 2:
        kbench.assertions.assert_true(False, expectation=f'Need >=2 confidence values. Got {len(confidences)}')
        return 0.0
    if expected_confidence_pattern == 'decreasing':
        slope = (confidences[-1] - confidences[0]) / max(len(confidences) - 1, 1)
        is_well_calibrated = slope <= 5.0
        kbench.assertions.assert_true(is_well_calibrated, expectation=f'Confidence slope should be <= 5pp/step. Was {slope:.1f}')
        return 1.0 if is_well_calibrated else 0.0
    return 1.0


In [ ]:
print(f"Evaluating test_confidence_calibration on {len(df)} items...")
with kbench.client.enable_cache():
    runs = test_confidence_calibration.evaluate(
        llm=MODELS_TO_TEST,
        evaluation_data=df,
    )


In [ ]:
try:
    df_res = runs.as_dataframe()
    df_res['score'] = df_res['result'].apply(lambda x: float(x) if x is not None else 0.0)
    plt.figure(figsize=(6, 5))
    pass_rate = df_res['score'].mean() * 100
    sns.barplot(x=['Pass', 'Fail'], y=[pass_rate, 100-pass_rate], palette=['#2ca02c', '#d62728'])
    plt.title('CCC: Epistemic Drift Resistance')
    plt.ylabel('% of Prompts')
    plt.show()
except Exception as e:
    print('Analytics error:', e)

%choose test_confidence_calibration
